In [1]:
# Enable inline plotting (required for Jupyter Notebooks)
%matplotlib inline

# Essential imports
import numpy as np
import pandas as pd
import statsmodels.api as sm
from uncertainties import unumpy
import math

In [2]:
# Function to compute asymmetric distance errors from predicted luminosity and observed flux

"""
This function computes the predicted distances and their asymmetric uncertainties 
based on gamma-ray luminosity and integral flux measurements. Since the 'uncertainties' 
module only supports symmetric error propagation, asymmetric uncertainties in the 
predicted luminosities (derived from the 68% prediction interval of the OLS model) 
are handled manually. Specifically, the upper and lower distance bounds are computed 
by substituting the upper and lower luminosity limits into the distance formula 
independently, while still using the flux uncertainties handled via the 'ufloat' 
representation. This approach ensures that asymmetric errors on luminosity are 
accurately reflected in the final distance predictions.
"""

def get_distance_pred_asymmetric(lumg_pred, lumg_pred_up, lumg_pred_down, flux, flux_error):
    cm2kpc = 3.24078e-22  # Conversion factor from cm to kpc

    # Create an array of flux values with uncertainties
    flux_prop = unumpy.uarray(flux, flux_error)
    flux_prop = 4 * math.pi * flux_prop

    # Compute upper and lower distance predictions from luminosity
    lum_up   = unumpy.uarray(lumg_pred, lumg_pred_up)
    lum_down = unumpy.uarray(lumg_pred, lumg_pred_down)

    dist_up = cm2kpc * unumpy.sqrt(lum_up / flux_prop)
    dist_down = cm2kpc * unumpy.sqrt(lum_down / flux_prop)

    return dist_up, dist_down

In [3]:
# --------------------------------------------------------
# Step 1: Read and prepare the training data for OLS model
# --------------------------------------------------------

# Define column names and widths based on the fixed-width file format (datafile9.txt)
col_names = [
    "Name", "f_Name", "LUMG", "P", "PdotCor", "EdotCor", "BdotS,Cor", 
    "ECut,SED", "n_ECut,SED", "ECut,B23", "n_ECut,B23", 
    "ECut,BFR", "n_ECut,BFR", "ECut_HYB", "n_ECut_HYB"
]
col_widths = [11, 6, 9, 7, 9, 9, 9, 8, 8, 6, 8, 5, 8, 6, 8]

# Read fixed-width data, skipping header lines
df = pd.read_fwf("datafile9.txt", names=col_names, widths=col_widths, skiprows=40)

# Filter pulsars marked with an asterisk (*) for training
df['f_Name'] = df['f_Name'].fillna('')
pc3_ols_df = df[df['f_Name'].str.contains(r'\*')].copy()

# Select relevant columns and strip whitespace from headers
keep_columns = ['LUMG', 'P', 'PdotCor', 'ECut_HYB']
pc3_ols_df.columns = pc3_ols_df.columns.str.strip()
pc3_ols_df = pc3_ols_df[keep_columns]

# Convert period to seconds
pc3_ols_df['P'] *= 1e-3

# Apply log10 transform to all numeric columns
for col in pc3_ols_df.columns:
    pc3_ols_df[col] = np.log10(pc3_ols_df[col])

# Split into predictors and target
y = pc3_ols_df['LUMG']
X = sm.add_constant(pc3_ols_df.drop(columns='LUMG'))
X = X[['const', 'P', 'PdotCor', 'ECut_HYB']]  # Ensure correct order

# Fit OLS model
model_ols = sm.OLS(y, X)
results_ols = model_ols.fit()

# Display summary of fit
print(results_ols.summary())
print("Parameters:", results_ols.params)
print("Standard errors:", results_ols.bse)

                            OLS Regression Results                            
Dep. Variable:                   LUMG   R-squared:                       0.840
Model:                            OLS   Adj. R-squared:                  0.835
Method:                 Least Squares   F-statistic:                     180.4
Date:                Fri, 11 Apr 2025   Prob (F-statistic):           7.49e-41
Time:                        19:59:41   Log-Likelihood:                -34.361
No. Observations:                 107   AIC:                             76.72
Df Residuals:                     103   BIC:                             87.41
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         40.5053      0.524     77.317      0.0

In [4]:
# ------------------------------------------------------------------
# Step 2: Apply the model to predict distances of unknown RQ pulsars
# ------------------------------------------------------------------

# Define column names and widths for datafile10_updated.txt
col_names_unk = [
    "Name", "ECut_HYB", "n_ECut_HYB", "P", "PdotCor", "EdotCor",
    "G100", "e_G100", "DOLS", "E_DOLS", "e_DOLS", "DRBF", "DSVR_RBF", "DSVR_LIN"
]
col_widths_unk = [10, 6, 7, 7, 9, 8, 9, 9, 5, 5, 5, 5, 5, 5]

# Read data for pulsars with unknown distances
df_unk = pd.read_fwf("datafile10.txt", names=col_names_unk, widths=col_widths_unk, skiprows=34)
df_unk.columns = df_unk.columns.str.strip()

# Extract relevant parameters
flux = df_unk['G100']
flux_err = df_unk['e_G100']
psr_name = df_unk['Name']
features = df_unk[['P', 'PdotCor', 'ECut_HYB', 'G100', 'e_G100']].copy()

# Convert period to seconds
features['P'] *= 1e-3

# Apply log10 to all but flux and its error
for col in ['P', 'PdotCor', 'ECut_HYB']:
    features[col] = np.log10(features[col])

# Prepare input for prediction
X_target = sm.add_constant(features.drop(columns=['G100', 'e_G100']))
X_target = X_target[['const', 'P', 'PdotCor', 'ECut_HYB']]

# Get predictions with 1-sigma confidence intervals (~68% or alpha=0.32)
prediction = results_ols.get_prediction(X_target)
pred_summary = prediction.summary_frame(alpha=0.32)

log_lumg = pred_summary["mean"]
log_lumg_up = pred_summary["obs_ci_upper"]
log_lumg_down = pred_summary["obs_ci_lower"]
log_lumg_err = log_lumg_up - log_lumg

# Convert to linear luminosities with uncertainties
log_lumg_u = unumpy.uarray(log_lumg, log_lumg_err)
lumg = np.power(10, log_lumg)
lumg_up = np.power(10, log_lumg_up)
lumg_down = np.power(10, log_lumg_down)
lumg_err_up = lumg_up - lumg
lumg_err_down = lumg - lumg_down

In [5]:
# ------------------------------------------------------------
# Step 3: Compute distances and distance errors for RQ pulsars
# ------------------------------------------------------------

# Compute distances
dist_up, dist_down = get_distance_pred_asymmetric(
    lumg, lumg_err_up, lumg_err_down,
    flux.values, flux_err.values
)

# Store final values
dist_mean = [d.nominal_value for d in dist_up]
dist_err_up = [d.std_dev for d in dist_up]
dist_err_down = [d.std_dev for d in dist_down]
    
# Print predicted distances with asymmetric errors in a clean table format
print(f"{'Pulsar':<20}{'Distance [kpc]':>20}{'+Error':>12}{'-Error':>12}")
print("-" * 64)

for name, d, err_up, err_down in zip(psr_name, dist_mean, dist_err_up, dist_err_down):
    print(f"{name:<20}{d:>20.2f}{err_up:>12.2f}{err_down:>12.2f}")

Pulsar                    Distance [kpc]      +Error      -Error
----------------------------------------------------------------
J0357+3205                          0.46        0.29        0.13
J0359+5414                          4.74        2.88        1.31
J0554+3107                          1.84        1.14        0.51
J0622+3749                          1.10        0.69        0.31
J0633+0632                          1.40        0.86        0.39
J0734-1559                          2.17        1.33        0.60
J0744-2525                          1.98        1.20        0.55
J0802-5613                          1.39        0.88        0.39
J1023-5746                          4.02        2.49        1.12
J1044-5737                          2.20        1.33        0.60
J1057-5851                          1.27        0.81        0.36
J1105-6037                          2.41        1.48        0.68
J1111-6039                          6.06        3.72        1.68
J1135-6055               